In [ ]:
# Cell 1: Import dependencies and define physical parameters
# This notebook solves the Zoeppritz equations for SV wave incidence
# at a solid-solid planar interface (Homework 5, Problem 1)

import numpy as np
from numpy.lib import scimath  # scimath.arcsin handles complex args beyond critical angles
import matplotlib.pyplot as plt

# --- Default physical parameters (geophysically reasonable) ---
# Medium 1: Shallow crust (upper crustal rock)
density_1 = 2.7       # rho_1 [g/cm^3]
p_velocity_1 = 5.8    # alpha_1 [km/s] - P-wave velocity
s_velocity_1 = 3.2    # beta_1  [km/s] - S-wave velocity

# Medium 2: Deep crust (lower crust, denser/faster)
density_2 = 3.3       # rho_2 [g/cm^3]
p_velocity_2 = 7.2    # alpha_2 [km/s]
s_velocity_2 = 3.9    # beta_2  [km/s]

print('Parameters loaded.')
print(f'Medium 1: rho={density_1}, vp={p_velocity_1}, vs={s_velocity_1}')
print(f'Medium 2: rho={density_2}, vp={p_velocity_2}, vs={s_velocity_2}')

In [ ]:
# Cell 2: Core Zoeppritz solver for SV wave incidence

def calculate_zoeppritz_sv(incident_angle_sv, density_1, p_velocity_1,
                            s_velocity_1, density_2, p_velocity_2,
                            s_velocity_2):
    """
    Compute reflection and transmission coefficients for an incident SV wave
    at a solid-solid planar interface using the Zoeppritz equations.
    
    Parameters
    ----------
    incident_angle_sv : float
        Incident SV wave angle [degrees]
    density_1, p_velocity_1, s_velocity_1 : float
        Density [g/cm^3], P-wave and S-wave velocities [km/s] in medium 1
    density_2, p_velocity_2, s_velocity_2 : float
        Density [g/cm^3], P-wave and S-wave velocities [km/s] in medium 2
    
    Returns
    -------
    dict : {
        'Rsp': reflected P-wave coefficient (complex),
        'Rss': reflected SV-wave coefficient (complex),
        'Tsp': transmitted P-wave coefficient (complex),
        'Tss': transmitted SV-wave coefficient (complex),
        'theta_P1': reflected P angle [rad],
        'theta_P2': transmitted P angle [rad],
        'theta_S2': transmitted SV angle [rad],
        'ray_parameter': ray parameter p
    }
    """
    # Convert incidence angle to radians
    i_rad = np.deg2rad(incident_angle_sv)
    
    # === Step 1: Snell's Law ===
    # Ray parameter p = sin(i) / beta_1 (constant across all waves)
    p = np.sin(i_rad) / s_velocity_1
    
    # === Step 2: Compute other wave angles via Snell's law ===
    # Use scimath.arcsin to handle complex angles beyond critical incidence
    # Reflected P-wave angle (mode-converted from incident SV)
    theta_P1 = scimath.arcsin(p * p_velocity_1)
    # Transmitted P-wave angle
    theta_P2 = scimath.arcsin(p * p_velocity_2)
    # Transmitted SV-wave angle
    theta_S2 = scimath.arcsin(p * s_velocity_2)
    # Reflected SV angle (= incident angle, by reflection law)
    theta_S1 = i_rad
    
    # === Step 3: Build the Zoeppritz matrix equation M * x = b ===
    # Following Aki & Richards (2002, eqn 5.37) for SV wave incidence
    #
    # Notation:
    #   th_f  = theta_P1  (reflected P)
    #   th_d  = theta_S1  (incident/reflected SV)
    #   th_f' = theta_P2  (transmitted P)
    #   th_d' = theta_S2  (transmitted SV)
    #
    # The unknown vector x = [Rsp, Rss, Tsp, Tss]^T
    
    # Short names for readability
    r1, a1, b1 = density_1, p_velocity_1, s_velocity_1
    r2, a2, b2 = density_2, p_velocity_2, s_velocity_2
    
    sin_f  = np.sin(theta_P1)
    cos_f  = np.cos(theta_P1)
    sin_d  = np.sin(theta_S1)
    cos_d  = np.cos(theta_S1)
    sin_f2 = np.sin(theta_P2)
    cos_f2 = np.cos(theta_P2)
    sin_d2 = np.sin(theta_S2)
    cos_d2 = np.cos(theta_S2)
    
    # Common sub-expressions: 1 - 2*beta^2*p^2 = cos(2*theta_S) for the SV angle
    term1_1 = 1.0 - 2.0 * b1**2 * p**2   # for medium 1
    term2_1 = 1.0 - 2.0 * b2**2 * p**2   # for medium 2
    
    # --- Build the 4x4 coefficient matrix M ---
    # Row 0: Vertical displacement (w) continuity
    #   w_incident + w_reflected = w_transmitted
    M = np.array([
        [-sin_f,    -cos_d,    -sin_f2,    cos_d2 ],  # w continuity
        [ cos_f,    -sin_d,     cos_f2,    sin_d2 ],  # u continuity
        [ 2*r1*b1**2*p*cos_f,  r1*b1*term1_1,  2*r2*b2**2*p*cos_f2,  -r2*b2*term2_1 ],  # sigma_zz
        [ -r1*a1*term1_1,  2*r1*b1**2*p*cos_d,  r2*a2*term2_1,  2*r2*b2**2*p*cos_d2 ]   # sigma_xz
    ])
    
    # --- Build the right-hand side vector b ---
    # Represents the incident SV wave contributions
    b = np.array([
        sin_d,              # w continuity: incident SV contributes sin(i)
        cos_d,              # u continuity: incident SV contributes cos(i)
        -r1 * b1 * term1_1,  # sigma_zz: incident SV contribution
        2 * r1 * b1**2 * p * cos_d  # sigma_xz: incident SV contribution
    ])
    
    # === Step 4: Solve the linear system ===
    x = np.linalg.solve(M, b)
    
    # Return results as a dictionary for clarity
    return {
        'Rsp': x[0],        # Reflected P-wave coefficient
        'Rss': x[1],        # Reflected SV-wave coefficient
        'Tsp': x[2],        # Transmitted P-wave coefficient
        'Tss': x[3],        # Transmitted SV-wave coefficient
        'theta_P1': theta_P1,
        'theta_P2': theta_P2,
        'theta_S2': theta_S2,
        'ray_parameter': p
    }

# Quick test: normal incidence (i=0)
result = calculate_zoeppritz_sv(0, density_1, p_velocity_1, s_velocity_1,
                                density_2, p_velocity_2, s_velocity_2)
print(f'Test at i=0°: Rsp={result["Rsp"]:.4f}, Rss={result["Rss"]:.4f}, '
      f'Tsp={result["Tsp"]:.4f}, Tss={result["Tss"]:.4f}')
print('Function ready.')

In [ ]:
# Cell 3: Batch computation across all incident angles and plot results

# --- Batch computation ---
angle_array = np.arange(0, 91, 0.5)  # 0 to 90 degrees, 0.5 degree steps
n_angles = len(angle_array)

# Pre-allocate arrays for coefficients (will be complex)
Rsp_arr = np.zeros(n_angles, dtype=complex)
Rss_arr = np.zeros(n_angles, dtype=complex)
Tsp_arr = np.zeros(n_angles, dtype=complex)
Tss_arr = np.zeros(n_angles, dtype=complex)

# Compute coefficients for each incident angle
for idx, angle in enumerate(angle_array):
    result = calculate_zoeppritz_sv(angle, density_1, p_velocity_1,
                                    s_velocity_1, density_2, p_velocity_2,
                                    s_velocity_2)
    Rsp_arr[idx] = result['Rsp']
    Rss_arr[idx] = result['Rss']
    Tsp_arr[idx] = result['Tsp']
    Tss_arr[idx] = result['Tss']

# --- Plotting ---
# --- Plotting ---
# Beyond critical angles, coefficients become complex.
# We plot the absolute value (amplitude) of each coefficient.

# Compute y-axis limits dynamically from data
all_vals = np.concatenate([np.abs(Rsp_arr), np.abs(Rss_arr),
                           np.abs(Tsp_arr), np.abs(Tss_arr)])
y_max = np.max(all_vals)
y_lim = y_max * 1.05  # 5% headroom

plt.figure(figsize=(10, 7))
plt.plot(angle_array, np.abs(Rsp_arr), 'b-',  linewidth=2, label=r'$|R_{sp}|$ (Reflected P)')
plt.plot(angle_array, np.abs(Rss_arr), 'r-',  linewidth=2, label=r'$|R_{ss}|$ (Reflected SV)')
plt.plot(angle_array, np.abs(Tsp_arr), 'g--', linewidth=2, label=r'$|T_{sp}|$ (Transmitted P)')
plt.plot(angle_array, np.abs(Tss_arr), 'm--', linewidth=2, label=r'$|T_{ss}|$ (Transmitted SV)')

# Mark critical angles with vertical lines
# Critical angle for reflected P-wave: sin(i_c) = beta_1 / alpha_1
crit_P1 = np.rad2deg(np.arcsin(s_velocity_1 / p_velocity_1))
# Critical angle for transmitted P-wave: sin(i_c) = beta_1 / alpha_2
crit_P2 = np.rad2deg(np.arcsin(s_velocity_1 / p_velocity_2))
# Critical angle for transmitted SV-wave: sin(i_c) = beta_1 / beta_2
crit_S2 = np.rad2deg(np.arcsin(s_velocity_1 / s_velocity_2))

for angle, label, color in [(crit_P1, r'$i_{c}^{P1}$', 'cyan'),
                            (crit_P2, r'$i_{c}^{P2}$', 'lime'),
                            (crit_S2, r'$i_{c}^{S2}$', 'orange')]:
    if not np.isnan(angle):
        plt.axvline(angle, color=color, linestyle=':', alpha=0.6)
        plt.text(angle + 0.5, y_lim * 0.9, label, color=color, fontsize=9)

plt.xlabel('Incident Angle (degrees)', fontsize=12)
plt.ylabel('Amplitude of Coefficients', fontsize=12)
plt.title('Zoeppritz Coefficients for Incident SV Wave', fontsize=14)
plt.legend(fontsize=10, loc='best')
plt.grid(True, alpha=0.3)
plt.xlim(0, 90)
plt.ylim(0, y_lim)

# Save figure to file
plt.savefig('figures/HW5_Q1_Zoeppritz.png', dpi=150, bbox_inches='tight')
print(f'Figure saved: figures/HW5_Q1_Zoeppritz.png')
print(f'Y-axis range: 0 to {y_lim:.2f}')
print(f'Critical angles (degrees):')
print(f'  Reflected P-wave  (sv→P1):  {crit_P1:.2f}°')
print(f'  Transmitted P-wave (sv→P2): {crit_P2:.2f}°')
print(f'  Transmitted SV-wave (sv→S2): {crit_S2:.2f}°')
import os as _os
if _os.environ.get('DISPLAY') or _os.name == 'nt':
    plt.show()
else:
    plt.close()